In [ ]:
"""
BigQuant 端到端 1分钟 ResidualAlpha V4 —— 官方提交入口

提交文件（4 个）：
1. ResidualAlphaV4_modelsave_predict.ipynb   本文件，提交包里唯一的 notebook
2. residual_alpha_v4_train_local.py          训练 / 推理共享脚本
3. residual_alpha_v4_model.py                模型结构与损失
4. residual_alpha_v4_model.json              本地训练后生成的权重（含多份 EMA 快照）

V4 = V3 的模型骨架 + 一批健壮性改造：
  * 字段归一化门控下限抬到 0.80，并排除出 weight decay（V3 的门控实际没在动）
  * 按验证目标取 Top-K epoch 的 EMA 权重做集成
  * 交易日骨架 = 成分股表 ∪ 注入分钟表的 DISTINCT 交易日，成分股表缺日也不丢日
  * 单日推理失败不再中断整段评估，交由后处理回填
  * 跨日轻度平滑，取向 IC_IR / SR
"""

import os

import numpy as np
import pandas as pd

# 严格在 Notebook 顶层导入：平台加载时完成绑定，main 被调用时不依赖 cwd / sys.path
from residual_alpha_v4_train_local import MODEL_PATH, predict_scores, train_and_save


def main(datasources, start_date, end_date):
    """平台调用入口，严格返回 date / instrument / score 三列。"""
    if not isinstance(datasources, dict) or not datasources.get("bar1m"):
        raise ValueError("datasources 必须包含 bar1m")

    result = predict_scores(
        datasources=datasources,
        start_date=start_date,
        end_date=end_date,
        model_path=MODEL_PATH,
    )

    required = ["date", "instrument", "score"]
    if not isinstance(result, pd.DataFrame):
        raise TypeError(f"main 必须返回 DataFrame，实际为 {type(result)}")
    missing = [c for c in required if c not in result.columns]
    if missing:
        raise ValueError(f"输出缺少字段：{missing}")

    result = result.loc[:, required].copy()
    result["date"] = pd.to_datetime(result["date"], errors="coerce").dt.normalize()
    result["instrument"] = result["instrument"].astype(str).str.strip()
    result["score"] = pd.to_numeric(result["score"], errors="coerce")
    result = result.replace([np.inf, -np.inf], np.nan).dropna(subset=required)

    start_day = pd.Timestamp(start_date).normalize()
    end_day = pd.Timestamp(end_date).normalize()
    result = result[result["date"].between(start_day, end_day)]
    result = (
        result.drop_duplicates(["date", "instrument"], keep="last")
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )
    if result.empty:
        raise RuntimeError("推理结果为空")

    # 交付前最后一道自检：截面不能退化成常数，否则平台做 z-score 会得到 NaN
    per_day = result.groupby("date")["score"]
    degenerate = per_day.std().fillna(0.0) < 1e-9
    if bool(degenerate.any()):
        raise RuntimeError(
            f"存在常数截面的交易日: {list(degenerate[degenerate].index[:5])}"
        )
    print(
        f"[main] {result['date'].nunique()} 个交易日 / {len(result)} 行 / "
        f"日均 {per_day.size().mean():.0f} 只"
    )
    return result


# 仅在显式开启时本地预览，避免任何后台脚本执行模式误触发
if __name__ == "__main__" and os.environ.get("BIGQUANT_LOCAL_PREVIEW") == "1":
    from bigmodule import M

    _start_date = "2024-01-02 00:00:00"
    _end_date = "2024-01-31 23:59:59"
    _score_data = main(
        {"bar1m": "bigalpha_2026_stock_bar1m"},
        _start_date,
        _end_date,
    )
    print(_score_data.head())
    M.bigalpha_eval._latest(
        factor_data=_score_data,
        start_date=_start_date,
        end_date=_end_date,
        show=True,
    )
